In [ ]:
import torch
import torchvision
import random
import matplotlib.pyplot as plt



In [ ]:
from torch.utils.data import Dataset,DataLoader
from torch import nn
from torchvision import transforms
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import random_split
from torch.nn.modules.pooling import MaxPool2d
from tqdm.auto import tqdm
from torch.utils.data import Subset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
device="cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
#import and extract zipfile
import zipfile

drive_loc=Path("/content/drive/MyDrive")
zip_path=drive_loc/"Refined_gesture_cropped.zip"
extract_to=Path("/content/Gesture")

if not extract_to.is_dir():
  extract_to.mkdir(parents=True , exist_ok=True)
else:
  print("It exists")

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_to)

In [ ]:
parent_dir=Path("Gesture/Refined_gesture_cropped")
subjects=["no_gesture","palm","one","peace","thumb_index","ok","three"]

In [ ]:
train_transform=transforms.Compose(
    [
        transforms.Resize(size=(128,128)),
        transforms.TrivialAugmentWide(num_magnitude_bins=15),
        transforms.RandomAffine(
              degrees=0,
              scale=(0.8, 1.2)   # zoom out to 80%, zoom in to 120%
),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor()
    ]

)

test_transform=transforms.Compose(
    [
        transforms.Resize(size=(128,128)),
        transforms.ToTensor()
    ]
)

In [ ]:
def helper_class(subjects):
  classes=subjects

  class_to_idx_dict={class_name:i for i,class_name in enumerate(classes)}

  return classes,class_to_idx_dict

In [ ]:
class Imagegesture(Dataset):
  def __init__(self,targ_dir,subjects,transform=None,):
    self.paths=[]
    self.transform=transform
    for subject in subjects:
      self.paths.extend(sorted(list( (targ_dir/subject).glob("*.jpg"))))

    self.classes,self.class_to_idx=helper_class(subjects)
  def __len__(self):
    return len(self.paths)

  def _loadimg(self,index):
    img_path=self.paths[index]
    return Image.open(img_path)

  def __getitem__(self, index):

    img=self._loadimg(index)
    class_name=self.paths[index].parent.stem



    if self.transform is not None:
      return self.transform(img),self.class_to_idx[class_name]

    else:
      return img,self.class_to_idx[class_name]








In [ ]:
dataset_1=Imagegesture(targ_dir=parent_dir,transform=train_transform,subjects=subjects)
dataset_2=Imagegesture(targ_dir=parent_dir,transform=test_transform,subjects=subjects)
class_names=dataset_1.classes
class_names

['no_gesture', 'palm', 'one', 'peace', 'thumb_index', 'ok', 'three']

In [ ]:
indices=list(range(len(dataset_1)))

train_idx,test_idx=train_test_split(indices,test_size=0.2,random_state=42)

In [ ]:
train_dataset=Subset(dataset_1,train_idx)
test_dataset=Subset(dataset_2,test_idx)

In [ ]:

Batch_size=32
train_dataloader=DataLoader(train_dataset, Batch_size,shuffle=True)
test_dataloader=DataLoader(test_dataset,Batch_size,shuffle=True)


In [ ]:
class Hand_gesture(nn.Module):
  def __init__(self,input_features,hidden_units,output_features):
    super().__init__()  #to make sure parent class creates necessary variables
    self.conv_layer_1=nn.Sequential(
        nn.Conv2d(in_channels=input_features,out_channels=hidden_units,kernel_size=3,padding=1,stride=1),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,kernel_size=3,padding=1,stride=1),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.conv_layer_2=nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,kernel_size=3,padding=1,stride=1),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,kernel_size=3,padding=1,stride=1),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.conv_layer_3=nn.Sequential(
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,kernel_size=3,padding=1,stride=1),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.Conv2d(in_channels=hidden_units,out_channels=hidden_units,kernel_size=3,padding=1,stride=1),
        nn.BatchNorm2d(hidden_units),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2,stride=2)
    )
    self.Linear_layer=nn.Sequential(
        nn.Flatten(),
        nn.Linear(
            in_features=hidden_units*256,
            out_features=output_features
        )

    )
  def forward(self,X):
    x=self.conv_layer_1(X)
    x=self.conv_layer_2(x)
    x=self.conv_layer_3(x)
    x=self.Linear_layer(x)
    return x

In [ ]:
def accuracy_fn(Y_pred,Y_true):
  count=torch.eq(Y_pred,Y_true).sum()
  percentage=(count/len(Y_pred))*100
  return percentage

In [ ]:

def train_step(model,dataloader,loss_fn,accuracy_fn,optimizer,device):
  model.train()
  model.to(device)
  avg_loss,avg_accuracy=0,0


  for X_train,Y_train in tqdm(dataloader):
    X_train,Y_train=X_train.to(device),Y_train.to(device)
    y_logits=model(X_train)
    loss=loss_fn(y_logits,Y_train)
    avg_loss+=loss.item()
    optimizer.zero_grad()
    accuracy=accuracy_fn(torch.softmax(y_logits,dim=1).argmax(dim=1),Y_train)
    avg_accuracy+=accuracy.item()
    loss.backward()
    optimizer.step()
  avg_loss/=len(dataloader)
  avg_accuracy/=len(dataloader)
  return avg_loss,avg_accuracy



In [ ]:
def test_step(model,dataloader,loss_fn,accuracy_fn,device):
  avg_loss,avg_accuracy=0,0
  model.eval()
  model.to(device)
  with torch.inference_mode():

    for X_test,Y_test in tqdm(dataloader):
      X_test,Y_test=X_test.to(device),Y_test.to(device)
      y_logits=model(X_test)
      accuracy=accuracy_fn(y_logits.argmax(dim=1),Y_test)
      avg_accuracy+=accuracy.item()
      loss=loss_fn(y_logits,Y_test)
      avg_loss+=loss.item()
  avg_loss/=len(dataloader)
  avg_accuracy/=len(dataloader)

  return avg_loss,avg_accuracy


In [ ]:


model_1=Hand_gesture(input_features=3,hidden_units=32,output_features=7)



In [ ]:
dummy_set=next(iter(train_dataloader))
dummy_set[0].shape

print(model_1(dummy_set[0]).argmax(dim=1))

tensor([5, 3, 3, 2, 3, 3, 1, 3, 3, 5, 2, 1, 3, 3, 3, 5, 5, 3, 3, 5, 5, 2, 3, 2,
        2, 5, 3, 5, 5, 3, 3, 3])


In [ ]:
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(params=model_1.parameters(),lr=0.0000001)


In [ ]:
epochs=10

for epoch in range(epochs):
  train_loss,train_accuracy=train_step(model=model_1,loss_fn=loss_fn,optimizer=optimizer,accuracy_fn=accuracy_fn,device=device,dataloader=train_dataloader)
  test_loss,test_accuracy=test_step(model=model_1,dataloader=test_dataloader,loss_fn=loss_fn,accuracy_fn=accuracy_fn,device=device)
  print("Train_loss:",train_loss,"Test_loss:",test_loss)
  print("Train_accuracy:",train_accuracy,"Test_accuracy:",test_accuracy)

  0%|          | 0/161 [00:00<?, ?it/s]

  0%|          | 0/41 [00:00<?, ?it/s]

Train_loss: 0.5504692021919333 Test_loss: 0.18846787935656653
Train_accuracy: 82.37577639751552 Test_accuracy: 95.04573170731707


  0%|          | 0/161 [00:00<?, ?it/s]

  0%|          | 0/41 [00:00<?, ?it/s]

...

  0%|          | 0/41 [00:00<?, ?it/s]

Train_loss: 0.5543124346259218 Test_loss: 0.18958342093520048
Train_accuracy: 82.00698757763975 Test_accuracy: 94.89329268292683


  0%|          | 0/161 [00:00<?, ?it/s]

  0%|          | 0/41 [00:00<?, ?it/s]

Train_loss: 0.5245350917442615 Test_loss: 0.191046498897599
Train_accuracy: 82.6086956521739 Test_accuracy: 95.04573170731707


In [ ]:

model_path = Path("models")
model_path.mkdir(parents=True, exist_ok=True)   # Create only the folder

modelpart_path = model_path / "model1best_83_95.pth"      # This is just a file path

torch.save(model_1.state_dict(), modelpart_path)

In [ ]:
model_1.eval()

for i in range(10):
    img, label = test_dataset[i]

    with torch.inference_mode():
        logits = model_1(img.unsqueeze(0))
        pred = logits.argmax(dim=1).item()

    print(
        f"True: {class_names[label]:12}  Pred: {class_names[pred]}"
    )

True: one           Pred: one
True: ok            Pred: ok
True: one           Pred: one
True: palm          Pred: palm
True: ok            Pred: ok
True: one           Pred: one
True: one           Pred: one
True: no_gesture    Pred: no_gesture
True: three         Pred: three
True: three         Pred: three
